In [17]:
import mlflow
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

mlflow.autolog() # <- Tutaj właczamy "magiczny" proces autologowania

# Ładowanie danych
titanic = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Przetwarzanie danych
titanic.drop(['Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
# titanic['Age'].fillna(titanic['Age'].median(), inplace=True)
titanic.fillna({"Age": titanic["Age"].median()}, inplace=True)
# titanic['Embarked'].fillna(titanic['Embarked'].mode()[0], inplace=True)
titanic.fillna({'Embarked': titanic['Embarked'].mode()[0]}, inplace=True)
titanic = pd.get_dummies(titanic, drop_first=True)

X = titanic.drop('Survived', axis=1)
y = titanic['Survived']

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Logujemy informację o użytych datasetach
# mlflow_dataset = mlflow.data.from_pandas(titanic, targets = "Survived",
#                                   name = "Titanic Dataset")
# mlflow.log_input(mlflow_dataset, context = "training")

mlflow_dataset = mlflow.data.from_pandas(
    titanic.drop(columns="Survived"),
    targets=titanic["Survived"].to_numpy(),
    name="Titanic Dataset"
)
mlflow.log_input(mlflow_dataset, context="training")

# Podział na zestawy treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Rozpoczęcie nowego eksperymentu
mlflow.set_experiment("Analiza danych Titanic")

# Definicja hiperparametrów do strojenia
param_grid = {
    'C': [0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}

# Inicjalizacja modelu
logreg = LogisticRegression(max_iter=200)

# Ustawienie GridSearchCV
grid_search = GridSearchCV(logreg, param_grid, cv=3, n_jobs=-1)

grid_search.fit(X_train, y_train)

# Najlepszy model
best_model = grid_search.best_estimator_

# Przewidywanie i ewaluacja
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

2025/07/28 18:36:42 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
/var/folders/r9/rrn82r812j9fccz3mm0wxk5w0000gp/T/ipykernel_2712/398157843.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  titanic['Age'].fillna(titanic['Age'].median(), inplace=True)
/var/folders/r9/rrn82r812j9fccz3mm0wxk5w0000gp/T/ipykernel_2712/398157843.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0

Fitting 5 folds for each of 108 candidates, totalling 540 fits


2025/07/28 18:36:48 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: RESOURCE_DOES_NOT_EXIST: Run '706e1dfe6b6d4eb98403987adb063390' not found


0.8156424581005587


In [3]:
y_pred

array([0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0,
       0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0,
       0, 1, 1])